# Make the atlas fit the format needed for snakemake

In [1]:
suppressPackageStartupMessages({
    library(data.table)
    library(Matrix)
    library(dplyr)
    library(SingleCellExperiment)
    library(batchelor)
    library(scuttle)
    library(scater)
    library(scran)
    library(BiocParallel)
})

In [2]:
io = list()
io$wd = getwd()

In [3]:
# Load sce
sce = readRDS(file.path(io$wd,"embryo_sce.rds"))

In [4]:
# Get metadata, add columns needed for processing
meta = as.data.table(colData(sce))
meta = meta[,`:=`(pass_rnaQC=TRUE, doublet_call=FALSE, celltype=celltype_extended_atlas, stripped=FALSE, doublet=FALSE)]

In [5]:
# get umap
umap = fread(file.path(io$wd,"umap.csv"))
# Add umap and sizefactors to metadata
meta = meta[,idx:=.I] %>%
    merge(.,umap, by='cell') %>% 
    .[order(idx)]

In [6]:
# Subset to max 15000 cells per stage
keep = lapply(unique(meta$stage), function(x){
  if(x == "mixed_gastrulation"){
    return(c())
  } else if(sum(meta$stage == x) < 15000) {
    return(which(meta$stage == x))
  } else {
    hits = which(meta$stage == x)
    return(sample(hits, 15000))
  }
})
keep = do.call(c, keep)

In [7]:
length(keep)

[1] 171787

In [8]:
# Subset
meta = meta[keep]
sce = sce[,meta$cell]

In [9]:
# Check if meta corresponds with sce
summary(meta$cell == colnames(sce))

   Mode    TRUE 
logical  171787 

In [10]:
head(meta,2)
head(sce)

cell,sample,embryo_version,stage,stage_mixed_gastrulation_mapped,somite_count,anatomy,S_score,G2M_score,phase,⋯,celltype_PijuanSala2019_E85mapped_WOTdescendant,celltype_extended_atlas,pass_rnaQC,doublet_call,celltype,stripped,doublet,idx,umapX,umapY
<chr>,<int>,<fct>,<fct>,<fct>,<chr>,<chr>,<dbl>,<dbl>,<fct>,⋯,<fct>,<fct>,<lgl>,<lgl>,<fct>,<lgl>,<lgl>,<int>,<dbl>,<dbl>
cell_1,1,Original,E6.5,E6.5,PooledEPooled6Pooled.Pooled5Pooled,Pooled,0.1717105,0.4300413,G2M,⋯,Epiblast,Epiblast,TRUE,FALSE,Epiblast,FALSE,FALSE,1,7.441005,16.45105
cell_2,1,Original,E6.5,E6.5,PooledEPooled6Pooled.Pooled5Pooled,Pooled,0.7101008,0.1834096,S,⋯,Primitive Streak,Primitive Streak,TRUE,FALSE,Primitive Streak,FALSE,FALSE,2,8.233685,16.70441


class: SingleCellExperiment 
dim: 6 171787 
metadata(0):
assays(1): counts
rownames(6): ENSMUSG00000051951 ENSMUSG00000089699 ...
  ENSMUSG00000025902 ENSMUSG00000104328
rowData names(0):
colnames(171787): cell_1 cell_2 ... ext_cell_317054 ext_cell_333126
colData names(19): cell sample ... celltype_extended_atlas sizeFactor
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [11]:
sizeFactors(sce) = meta$sizeFactor

In [12]:
saveRDS(sce, file.path(io$wd, 'processed/SingleCellExperiment.rds'))

In [13]:
fwrite(meta, file.path(io$wd, 'sample_metadata.txt.gz'))